# UD1.03. Del dato crudo al análisis: JSON, XML, YAML y HTML

**Módulo 5073 · Programación de Inteligencia Artificial · UD1 · Práctica P1.2**

Ningún proyecto de inteligencia artificial empieza con un CSV limpio. Empieza con un API que
devuelve JSON, un organismo público que publica XML, una configuración en YAML y, a veces, una
tabla dentro de una página HTML.

En este cuaderno vas a recorrer los cuatro formatos sobre un mismo escenario: **el gasto público
en tecnología en la Comunitat Valenciana**. Al final tendrás los cuatro orígenes convertidos a
una única estructura de datos, lista para analizar en la UD3.

El enunciado completo, con los pesos de cada parte, está en la práctica
**P1.2 · Del dato crudo al análisis**.

---

## Cómo se corrige

Las celdas Markdown cuentan tanto como las de código. El criterio de evaluación que se mide
aquí es el **1.f**, *"se han caracterizado lenguajes de marcado destacando la información que
contienen sus etiquetas"*: no basta con conseguir leer los ficheros, hay que **explicar qué
dice cada etiqueta**.

Busca las celdas marcadas con `# COMPLETA` y `**Responde:**`. Son las que se evalúan.

## Preparación

Si trabajas en **Colab**, ejecuta la celda siguiente para descargar los ficheros de datos.
Si trabajas en **local**, salta la celda: los ficheros ya están en `datos/`.

In [ ]:
# Solo en Google Colab: descarga los ficheros de datos de la unidad
import os, urllib.request

BASE = ("https://raw.githubusercontent.com/RafaSalaEsteve/IABD-PIA-notebooks"
        "/main/UD1/datos/")
FICHEROS = ["licitaciones_api.json", "licitaciones_pcsp.xml", "portal_transparencia.html"]

if not os.path.exists("datos"):
    os.makedirs("datos", exist_ok=True)
    for f in FICHEROS:
        urllib.request.urlretrieve(BASE + f, os.path.join("datos", f))
        print("descargado:", f)
else:
    print("La carpeta datos/ ya existe, no descargo nada")

In [ ]:
!pip install -q pyyaml beautifulsoup4    # en Colab suelen estar ya instalados

---

# Parte 1 · JSON desde un API

**Peso: 25 %** · Criterio 1.f

JSON es el formato que devuelven casi todos los API web. Tiene solo seis tipos de datos
(cadena, número, booleano, `null`, array y objeto) y esa pobreza es justo lo que lo hace
universal.

In [ ]:
import json
from datetime import datetime
from pathlib import Path

RUTA_JSON = Path("datos/licitaciones_api.json")

with open(RUTA_JSON, encoding="utf-8") as f:
    respuesta = json.load(f)

print("Tipo del objeto raíz:", type(respuesta))
print("Claves de primer nivel:", list(respuesta.keys()))
print("Número de licitaciones:", len(respuesta["licitaciones"]))

In [ ]:
# Mira el primer registro entero antes de programar nada sobre él
print(json.dumps(respuesta["licitaciones"][0], indent=2, ensure_ascii=False))

### 1.1. Describe la estructura

**Responde en esta celda:**

1. ¿Qué partes del documento son **objetos** JSON y cuáles son **arrays**? ¿A qué profundidad
   está cada dato de una licitación?
2. Completa la tabla siguiente con **todas** las claves de la sección `meta` y de un registro
   de `licitaciones`.

| Clave | Nivel | Tipo JSON | Tipo Python | Qué información aporta |
|---|---|---|---|---|
| `meta` | 1 | object | `dict` | Metadatos de la respuesta, no datos de negocio |
| `meta.total` | 2 | number | `int` | Cuántos registros trae la respuesta |
| ... | | | | |

*(Escribe aquí tu respuesta.)*

In [ ]:
# COMPLETA
# Extrae de cada licitación: id, organo, objeto, importe y fechaLimite.
# La fecha viene como cadena en formato ISO 8601: conviértela a datetime.
#
# Pista: datetime.fromisoformat("2026-10-15T14:00:00")

def extrae_json(registro: dict) -> dict:
    ...


licitaciones_json = [extrae_json(r) for r in respuesta["licitaciones"]]
licitaciones_json[:2]

### 1.2. Valida antes de procesar

El fichero tiene **registros defectuosos a propósito**. Antes de usar los datos hay que
apartarlos, no dejar que rompan el análisis tres pasos más adelante.

Un registro es válido si cumple **todo** esto:

- Tiene las cinco claves obligatorias: `id`, `organo`, `objeto`, `importe`, `fechaLimite`.
- `importe` es un número (no una cadena) y es **mayor que cero**.
- `fechaLimite` se puede convertir a `datetime`.

Los que no cumplan van a una lista de rechazados **con el motivo**, para poder informar de qué
falló.

In [ ]:
# COMPLETA
OBLIGATORIAS = ["id", "organo", "objeto", "importe", "fechaLimite"]

def valida(registro: dict) -> str | None:
    """Devuelve None si el registro es válido, o el motivo del rechazo."""
    ...


validos, rechazados = [], []
for r in respuesta["licitaciones"]:
    motivo = valida(r)
    if motivo is None:
        validos.append(extrae_json(r))
    else:
        rechazados.append((r.get("id", "SIN ID"), motivo))

print(f"Válidos: {len(validos)}   Rechazados: {len(rechazados)}")
for id_, motivo in rechazados:
    print(f"  - {id_}: {motivo}")

**Responde:** ¿cuántos registros ha rechazado y por qué motivo cada uno? ¿Te parece que alguno
de los rechazos podría recuperarse en lugar de descartarse, y cómo?

*(Escribe aquí tu respuesta.)*

---

# Parte 2 · XML de la administración

**Peso: 30 %** · Criterio 1.f

`licitaciones_pcsp.xml` reproduce la estructura del canal Atom que publica la Plataforma de
Contratación del Sector Público. Es más verboso que JSON y tiene una dificultad que JSON no
tiene: **los espacios de nombres**.

> **Antes de programar nada, abre el fichero en un editor de texto y léelo.** Es el paso que
> más tiempo ahorra.

In [ ]:
# Mira las primeras líneas del fichero tal cual
print(Path("datos/licitaciones_pcsp.xml").read_text(encoding="utf-8")[:1400])

### 2.1. Anatomía del documento

**Responde en esta celda:**

1. ¿Cuál es el elemento **raíz**? ¿Cuántas raíces puede tener un documento XML bien formado?
2. ¿Qué **espacios de nombres** se declaran y para qué sirve cada uno?
3. Pon un ejemplo del documento de una **etiqueta** y otro de un **atributo**, y explica la
   diferencia entre ambos conceptos.
4. Explica **qué información aporta** cada una de estas etiquetas:
   `cbc:ContractFolderID`, `cbc:Name`, `cbc:TotalAmount`, `cbc:EndDate`.

*(Escribe aquí tu respuesta.)*

In [ ]:
import xml.etree.ElementTree as ET

arbol = ET.parse("datos/licitaciones_pcsp.xml")
raiz = arbol.getroot()

print("Etiqueta de la raíz:", raiz.tag)   # fíjate: viene con el espacio de nombres delante

# Diccionario de prefijos: es lo que permite usar rutas legibles en findall()
NS = {
    "atom": "http://www.w3.org/2005/Atom",
    "cbc":  "urn:dgpe:names:draft:codice:schema:xsd:CommonBasicComponents-2",
    "cac":  "urn:dgpe:names:draft:codice:schema:xsd:CommonAggregateComponents-2",
}

entradas = raiz.findall("atom:entry", NS)
print("Entradas encontradas:", len(entradas))

In [ ]:
# Ejemplo resuelto: extraer el identificador de la primera entrada
primera = entradas[0]
id_exp = primera.find(".//cbc:ContractFolderID", NS).text
print("Expediente:", id_exp)

In [ ]:
# COMPLETA
# Extrae de cada entry los mismos cinco campos que en la parte 1:
#   id      -> cbc:ContractFolderID
#   organo  -> el cbc:Name que está dentro de cac:PartyName
#   objeto  -> el cbc:Name que está dentro de cac:ProcurementProject
#   importe -> cbc:TotalAmount  (ojo: es texto, conviértelo a float)
#   fecha   -> cbc:EndDate + cbc:EndTime
#
# Cuidado: hay DOS elementos cbc:Name en cada entrada. Usa la ruta completa
# para no confundirlos.

def extrae_xml(entry) -> dict:
    ...


licitaciones_xml = [extrae_xml(e) for e in entradas]
licitaciones_xml

### 2.2. Bien formado y válido

**Responde en esta celda:**

1. Este documento es **bien formado**. ¿Qué significa exactamente y qué tres reglas cumple?
2. ¿Qué haría falta para poder decir además que es **válido**?
3. ¿Qué ventaja práctica tiene poder validar un documento contra un esquema antes de
   procesarlo? Piensa en un proceso que recibe 5.000 ficheros al día.

*(Escribe aquí tu respuesta.)*

---

# Parte 3 · YAML de configuración

**Peso: 15 %** · Criterio 1.f

Hasta ahora las rutas y los umbrales están escritos a mano en el código. Eso está mal: cada vez
que cambia algo hay que tocar el programa. La configuración va **fuera**, en un fichero que
puede editar alguien que no sabe programar.

YAML es el formato habitual para eso: admite comentarios, no obliga a comillas y la estructura
la marca la indentación.

In [ ]:
# COMPLETA
# Escribe el contenido de analisis.yaml. Debe incluir, al menos:
#   - las rutas de los tres ficheros de entrada
#   - la lista de campos obligatorios de la validación
#   - el importe mínimo para incluir una licitación en el análisis
#   - el formato de salida deseado
# Pon un comentario explicando cada bloque.

configuracion_yaml = """
# Configuración del análisis de gasto público en tecnología
# Módulo 5073 - UD1 - Práctica P1.2

entradas:
  # COMPLETA
"""

Path("analisis.yaml").write_text(configuracion_yaml, encoding="utf-8")
print(configuracion_yaml)

In [ ]:
import yaml

with open("analisis.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)     # safe_load, nunca load

print(config)

### 3.1. Los dos avisos de YAML

**a) Tipos que parecen otra cosa.** Ejecuta la celda siguiente y observa qué pasa cuando un
valor no lleva comillas.

In [ ]:
prueba = yaml.safe_load("""
respuesta_sin_comillas: no
respuesta_con_comillas: "no"
version_sin_comillas: 1.10
version_con_comillas: "1.10"
""")

for clave, valor in prueba.items():
    print(f"{clave:26} -> {valor!r:8} ({type(valor).__name__})")

**Responde:**

1. ¿Qué ha pasado con `no` y con `1.10`? ¿Por qué es un problema si esos valores fueran un
   código de país o el número de versión de una biblioteca?
2. ¿Por qué se usa `yaml.safe_load` y no `yaml.load`? ¿Qué puede ocurrir con `load` si el
   fichero viene de una fuente que no controlas?
3. ¿Por qué en YAML está prohibido el tabulador?

*(Escribe aquí tu respuesta.)*

In [ ]:
# COMPLETA
# A partir de aquí, ninguna ruta ni ningún umbral puede estar escrito a mano en el código:
# todo tiene que salir de `config`. Reescribe la carga del JSON usando la configuración.

---

# Parte 4 · HTML

**Peso: 20 %** · Criterio 1.f

La última vía es una página web. Se trabaja sobre un **fichero local** a propósito.

> Si en un proyecto real necesitas datos de una web: revisa antes su `robots.txt` y sus
> condiciones de uso, espacia las peticiones para no saturar el servidor, y comprueba si existe
> un API o un portal de datos abiertos que evite tener que raspar el HTML. Que un dato sea
> accesible no lo convierte en reutilizable: si contiene datos personales, se aplica el RGPD.

In [ ]:
from bs4 import BeautifulSoup

html = Path("datos/portal_transparencia.html").read_text(encoding="utf-8")
sopa = BeautifulSoup(html, "html.parser")

print(sopa.find("h1").get_text(strip=True))
print("Filas de la tabla:", len(sopa.select("#tabla-contratos tbody tr")))

In [ ]:
# Ejemplo resuelto: primera fila, celda a celda
fila = sopa.select_one("#tabla-contratos tbody tr")
for celda in fila.find_all("td"):
    print(repr(celda.get_text(strip=True)))

# El importe está dos veces: formateado para leer y en bruto en un atributo
importe = fila.select_one("td.importe")
print("Texto:", importe.get_text(strip=True), "| Atributo data-valor:", importe["data-valor"])

### 4.1. Marcado semántico frente a marcado de presentación

Esta página está escrita a propósito mezclando los dos estilos.

**Responde en esta celda:**

1. Localiza al menos **cuatro etiquetas semánticas** de la página y di qué significa cada una.
2. Localiza al menos **tres marcas de presentación** (etiquetas o clases que solo dicen cómo se
   ve algo) y explica por qué son peores.
3. Fíjate en `<td class="importe" data-valor="14800.00">14.800,00 EUR</td>`. ¿Por qué es mucho
   mejor leer el atributo `data-valor` que el texto visible? ¿Qué tendrías que hacer si solo
   dispusieras del texto?
4. ¿Qué relación tiene todo esto con la **accesibilidad**? Piensa en cómo recorre esta página
   un lector de pantalla.

*(Escribe aquí tu respuesta.)*

In [ ]:
# COMPLETA
# Extrae las cinco filas de la tabla a la misma estructura que en las partes 1 y 2.
# Usa data-valor para el importe y el atributo datetime para la fecha.

def extrae_html(fila) -> dict:
    ...


licitaciones_html = [extrae_html(f) for f in sopa.select("#tabla-contratos tbody tr")]
licitaciones_html

---

# Parte 5 · Unificación y conclusión

**Peso: 10 %**

Los cuatro orígenes tienen que acabar en **una sola estructura**, con las mismas claves y los
mismos tipos. Es exactamente lo que harás en cualquier proyecto real antes de poder analizar
nada.

In [ ]:
# COMPLETA
# 1. Junta las tres listas en una sola, añadiendo a cada registro una clave 'origen'
#    con el valor 'json', 'xml' o 'html'.
# 2. Comprueba que todos los registros tienen exactamente las mismas claves.
# 3. Comprueba que 'importe' es float y 'fechaLimite' es datetime en todos.

todas = ...

In [ ]:
# COMPLETA
# Calcula:
#   - el importe total
#   - el número de licitaciones y el importe acumulado por órgano, de mayor a menor
#   - la licitación con la fecha límite más próxima

### 5.1. Conclusión

**Responde:** si tuvieras que **publicar** este conjunto unificado para que lo consumieran
otros equipos, ¿en cuál de los cuatro formatos lo harías?

Argumenta con criterios técnicos, no por preferencia. Ten en cuenta al menos: quién lo va a
consumir (una persona o un programa), si necesita comentarios, si conviene poder validarlo
contra un esquema, y cuánto pesa cada formato para el mismo contenido.

*(Escribe aquí tu respuesta.)*

---

## Antes de entregar

- [ ] Ejecuta **Kernel → Restart & Run All**. Si falla, no está terminado.
- [ ] Todas las celdas `# COMPLETA` están resueltas.
- [ ] Todas las celdas `**Responde:**` tienen tu respuesta escrita.
- [ ] Ninguna ruta ni umbral está escrito a mano: todo sale de `analisis.yaml`.
- [ ] Renombra el fichero a `UD1_P1.2_Apellidos_Nombre.ipynb`.
- [ ] Si has usado un asistente de IA, decláralo al final indicando para qué.

Entrega por Aules, tarea **P1.2 · Del dato crudo al análisis**.